# QueryPilot

db + loader + charts + agent, all in one notebook. Run top to bottom.

In [1]:
# stdlib stuff — file paths, database, temp files, running the server in a thread
import os
import re
import shutil
import sqlite3
import tempfile
import threading
import asyncio

# third-party stuff
import pandas as pd      # reads csv/excel files into tables
import ollama            # talks to the local LLM (qwen3-coder)
import uvicorn            # the web server that runs our FastAPI app
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel   # used to define what a valid /ask request looks like

## db

Building the sample database, reading its schema, running queries. Plain sqlite3.

In [2]:
# Path to the db file and the script used to seed it
DB_PATH = "querypilot.db"
SEED_PATH = "seed.sql"


def ensure_db_file():
    # wipe and start fresh every time the server boots — no leftover tables
    # from a previous run, so you always start clean and re-upload your data
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
    sqlite3.connect(DB_PATH).close()


def load_demo_data():
    # reads seed.sql (plain CREATE TABLE + INSERT statements) as text,
    # then executescript() runs the whole file in one go against the db.
    # seed.sql starts with DROP TABLE IF EXISTS, so this is safe to call
    # again even if tables already exist — it just wipes and rebuilds them.
    with open(SEED_PATH, "r") as f:
        script = f.read()

    conn = sqlite3.connect(DB_PATH)
    conn.executescript(script)
    conn.commit()
    conn.close()


def get_connection():
    # every function that needs the db opens its own connection and closes
    # it when done, instead of sharing one connection everywhere
    return sqlite3.connect(DB_PATH)


def get_schema(conn):
    # sqlite keeps every table's original CREATE TABLE statement in a system
    # table called sqlite_master — this just reads them out. this text is
    # what gets pasted into the prompt so the model knows what columns exist
    cur = conn.execute(
        "SELECT sql FROM sqlite_master WHERE type='table' AND sql IS NOT NULL"
    )
    tables = [row[0] for row in cur.fetchall()]
    return "\n\n".join(tables)


def run_query(conn, sql):
    # runs whatever SQL string it's given. if the SQL is bad (wrong column
    # name, typo, etc) sqlite3 raises an error on its own — we don't have to
    # check anything ourselves, the agent's retry loop catches that error
    cur = conn.execute(sql)
    rows = cur.fetchall()
    # cur.description is None for queries with no result columns (rare here)
    columns = [d[0] for d in cur.description] if cur.description else []
    return rows, columns

## loader

Load a CSV or Excel file into the database as a new table.

In [3]:
def clean_name(name):
    # sql column/table names can't have spaces or symbols, so we turn
    # 'Order Date' -> 'order_date' and 'Total ($)' -> 'total'
    name = name.strip().lower()                    # 'Order Date' -> 'order date'
    name = re.sub(r"[^a-z0-9]+", "_", name)         # 'order date' -> 'order_date'
    return name.strip("_")                          # trims leading/trailing _


def load_file(path, table_name=None):
    # figures out the file type from its extension and reads it with pandas
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        frame = pd.read_csv(path)
    elif ext in (".xlsx", ".xls"):
        frame = pd.read_excel(path)
    else:
        raise ValueError(f"unsupported file type: {ext}")

    # clean up every column header so it's a safe sql name
    frame.columns = [clean_name(c) for c in frame.columns]
    # table gets named after the file itself, e.g. sales.csv -> table "sales".
    # table_name lets the caller pass the ORIGINAL filename — path might be a
    # temp file (upload_endpoint saves uploads under a random temp name)
    table = clean_name(table_name if table_name else os.path.splitext(os.path.basename(path))[0])

    # to_sql does the actual work: creates the table and inserts every row.
    # if_exists="replace" means re-uploading the same file just overwrites it
    conn = sqlite3.connect(DB_PATH)
    frame.to_sql(table, conn, if_exists="replace", index=False)
    conn.commit()
    conn.close()
    return table

## charts

Simple rules, no model needed. Looks at the shape of the result and the column names.

In [4]:
# column names that usually mean "time" -> line chart reads best
TIME_HINTS = ["date", "month", "year", "day", "time", "quarter", "week"]


def is_number(value):
    # bool is technically a subclass of int in python (True == 1), so we
    # explicitly exclude it — a True/False column shouldn't count as "numeric"
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def pick_chart(columns, rows):
    # no model involved here at all — just plain if/else rules based on the
    # shape of the result. returns one of: kpi, line, pie, bar, table
    if not rows or not columns:
        return "table"

    # one row, one column, and it's a number -> show it as a big kpi number
    # e.g. "how many orders?" -> just show "20" big on screen
    if len(rows) == 1 and len(columns) == 1 and is_number(rows[0][0]):
        return "kpi"

    # two columns where the second one is numeric -> label + value, chart it
    if len(columns) == 2 and is_number(rows[0][1]):
        label = columns[0].lower()
        # if the label column looks like a date/time, a line chart reads best
        if any(hint in label for hint in TIME_HINTS):
            return "line"
        # few rows -> pie chart is readable, many rows -> bar chart instead
        return "pie" if len(rows) <= 6 else "bar"

    # anything else (more columns, mixed types, etc) -> just show a table
    return "table"

## agent

Turns a plain-English question into SQL, checks it is safe, runs it, and retries
by feeding the error back to the model if the query fails.

    question -> write sql -> safety check -> run -> (error? retry) -> rows

In [5]:
# Local model used to write SQL, override with QP_MODEL env var if needed
MODEL = os.environ.get("QP_MODEL", "qwen3-coder:30b")

MAX_RETRIES = 2   # how many extra tries after the first attempt fails

# only read-only SELECTs are allowed, block anything that changes data
BANNED_WORDS = [
    "insert", "update", "delete", "drop", "alter",
    "create", "replace", "attach", "pragma",
]


def is_safe(sql):
    # this is the security check — makes sure the model can't sneak in
    # anything destructive, even if the prompt somehow gets it to try
    q = sql.strip().lower()

    # must start with select or with (a CTE), nothing else allowed
    if not (q.startswith("select") or q.startswith("with")):
        return False

    # blocks "SELECT 1; DROP TABLE x" — a semicolon anywhere except the very
    # end would mean there's a second statement hiding after it
    if ";" in q.rstrip(";"):
        return False

    # simple keyword blacklist — if any dangerous word shows up anywhere
    # in the query, reject it outright
    for word in BANNED_WORDS:
        if word in q:
            return False

    return True


def clean_sql(text):
    # the model sometimes wraps its answer in ```sql ... ``` markdown or
    # adds a sentence of explanation before/after — this strips all that
    # down to just the raw SQL statement
    text = text.replace("```", " ")

    # find where "select" or "with" starts and cut off anything before it
    match = re.search(r"\b(select|with)\b", text, re.IGNORECASE)
    if match:
        text = text[match.start():]

    # cut off anything after the first semicolon (extra text, more statements)
    if ";" in text:
        text = text[:text.index(";")]

    return text.strip()


def generate_sql(schema, question, error=None, history=None):
    # builds the prompt that gets sent to the LLM
    prompt = (
        "You are a SQLite expert. Using the schema below, write ONE read-only "
        "SELECT query that answers the question. Return only the SQL, no "
        "explanation and no markdown.\n\n"
        f"Schema:\n{schema}\n\n"
    )

    # if there's chat history, include the last few turns so a follow-up
    # question like "what about the North region?" still makes sense
    if history:
        recent = history[-4:]
        turns = "\n".join(f"Q: {h['q']}\nSQL: {h['sql']}" for h in recent)
        prompt += f"Earlier in this chat:\n{turns}\n\n"

    prompt += f"Question: {question}\n"

    # this is the self-correction bit — if the previous attempt failed,
    # we tell the model exactly what error it got so it can fix itself
    if error:
        prompt += f"\nThe previous query failed with this error: {error}\nFix it."

    resp = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return clean_sql(resp["message"]["content"])


def ask(question, history=None):
    # this is the main pipeline: write sql -> check it's safe -> run it ->
    # if it fails, loop back and try again with the error included
    conn = get_connection()
    schema = get_schema(conn)

    # no tables yet (nothing uploaded) -> don't even bother calling the model
    if not schema:
        conn.close()
        return {"sql": "", "rows": None, "columns": [], "attempts": 0,
                "error": "no data loaded yet — upload a file first"}

    error = None
    sql = ""

    # tries once, then up to MAX_RETRIES more times if something goes wrong
    for attempt in range(MAX_RETRIES + 1):
        sql = generate_sql(schema, question, error, history)

        # security check failed -> stop immediately, don't even try running it
        if not is_safe(sql):
            conn.close()
            return {"sql": sql, "rows": None, "columns": [], "attempts": attempt + 1,
                    "error": "blocked: not a read-only query"}

        try:
            rows, columns = run_query(conn, sql)
            conn.close()
            return {"sql": sql, "rows": rows, "columns": columns,
                    "attempts": attempt + 1, "error": None}
        except Exception as e:
            error = str(e)   # save the error, loop goes back and retries

    # ran out of retries — return whatever the last error was
    conn.close()
    return {"sql": sql, "rows": None, "columns": [],
            "attempts": MAX_RETRIES + 1, "error": error}


def explain(question, columns, rows):
    # separate LLM call — turns the raw rows into a short plain-English
    # sentence instead of just dumping numbers on screen
    if not rows:
        return "No results found."

    # only send the first 30 rows to the model, no need to send everything
    preview = [columns] + [list(r) for r in rows[:30]]
    prompt = (
        "Answer the question in one or two short sentences based only on this "
        "result. Do not mention SQL.\n\n"
        f"Question: {question}\n"
        f"Result: {preview}\n"
    )
    resp = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()

## self-checks

Same asserts that used to live in each file's `if __name__ == "__main__"` block.

In [6]:
assert clean_name("Order Date") == "order_date"
assert clean_name("Total ($)") == "total"
assert clean_name("customer_id") == "customer_id"

assert pick_chart(["total"], [(500,)]) == "kpi"
assert pick_chart(["month", "sales"], [("2024-01", 10), ("2024-02", 20)]) == "line"
assert pick_chart(["region", "sales"], [("N", 5), ("S", 7)]) == "pie"
assert pick_chart(["name", "qty"], [(str(i), i) for i in range(10)]) == "bar"
assert pick_chart(["a", "b", "c"], [(1, 2, 3)]) == "table"

assert is_safe("SELECT * FROM orders")
assert is_safe("WITH t AS (SELECT 1) SELECT * FROM t")
assert not is_safe("DROP TABLE orders")
assert not is_safe("DELETE FROM customers")
assert not is_safe("SELECT 1; DROP TABLE orders")
assert not is_safe("update products set price = 0")

print("all checks passed")

all checks passed


## web backend

FastAPI app, serves the web/ page and exposes /ask and /upload.

In [7]:
# the actual web app object — every @app.get/@app.post below adds one route to it
app = FastAPI()
app.mount("/static", StaticFiles(directory="web"), name="static")   # web page assets


class Question(BaseModel):
    # pydantic checks incoming JSON matches this shape automatically —
    # if /ask gets a request missing "question", FastAPI rejects it for us
    question: str
    history: list[dict] = []   # earlier turns: {"q": ..., "sql": ...}


@app.on_event("startup")
def startup():
    # runs once when the server boots up
    ensure_db_file()   # empty db, waiting for the user to upload something


@app.get("/")
def home():
    # serves the chat page itself when you open http://127.0.0.1:8000
    return FileResponse("web/index.html")


@app.post("/ask")
def ask_endpoint(payload: Question):
    # this is what the chat UI calls every time you send a question
    result = ask(payload.question, payload.history)

    if result["error"]:
        return {
            "error": result["error"],
            "sql": result["sql"],
            "trace": ["wrote SQL", f"stopped: {result['error']}"],
        }

    # sqlite rows come back as tuples, convert to lists so they turn into
    # clean JSON arrays instead of python-specific tuple syntax
    rows = [list(r) for r in result["rows"]]
    answer = explain(payload.question, result["columns"], result["rows"])
    chart = pick_chart(result["columns"], result["rows"])

    # a little step-by-step trace shown in the "how it worked" dropdown in the UI
    trace = [
        "understood the question",
        "wrote SQL",
        "safety check passed (read-only)",
        f"ran query — {result['attempts']} attempt(s)",
    ]

    return {
        "error": None,
        "sql": result["sql"],
        "columns": result["columns"],
        "rows": rows,
        "answer": answer,
        "chart": chart,
        "trace": trace,
    }


def merge_db_file(path):
    # ATTACH lets one connection see two database files at once, under the
    # names "main" (ours) and "src" (the uploaded one). we copy each table
    # from src into main one at a time — this merges tables in instead of
    # replacing the whole file, so upload order never wipes earlier uploads
    conn = get_connection()
    conn.execute("ATTACH DATABASE ? AS src", (path,))

    tables = [row[0] for row in conn.execute(
        "SELECT name FROM src.sqlite_master WHERE type='table'"
    )]
    for t in tables:
        conn.execute(f"DROP TABLE IF EXISTS main.{t}")
        conn.execute(f"CREATE TABLE main.{t} AS SELECT * FROM src.{t}")
    conn.commit()

    conn.execute("DETACH DATABASE src")
    schema = get_schema(conn)
    conn.close()
    return f"{schema.count('CREATE TABLE')} table(s) loaded"


@app.post("/upload")
async def upload_endpoint(file: UploadFile = File(...)):
    # save the uploaded file to a temp path first, since load_file() needs
    # an actual file path on disk, not the raw upload bytes
    suffix = os.path.splitext(file.filename)[1].lower()
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name

    try:
        if suffix in (".db", ".sqlite", ".sqlite3"):
            # a whole database file was uploaded — merge its tables into
            # ours instead of replacing everything, so it stacks with
            # whatever csv/xlsx/sql files were already uploaded
            table = merge_db_file(tmp_path)
        elif suffix == ".sql":
            # a raw sql script (CREATE TABLE + INSERT statements) — same
            # trick load_demo_data() uses, just run it against the db
            with open(tmp_path, "r") as f:
                script = f.read()
            conn = get_connection()
            conn.executescript(script)
            conn.commit()
            schema = get_schema(conn)
            conn.close()
            table = f"{schema.count('CREATE TABLE')} table(s) loaded"
        else:
            # name the table after the ORIGINAL filename, not the random
            # temp filename it got saved under on disk
            original_name = os.path.splitext(file.filename)[0]
            table = load_file(tmp_path, table_name=original_name)
    finally:
        os.remove(tmp_path)   # clean up the temp file either way

    return {"table": table}


@app.post("/demo")
def demo_endpoint():
    # "Load demo data" button in the UI calls this
    load_demo_data()
    return {"table": "demo data loaded"}

/var/folders/y6/gjlj_p8n349gv21jdzdb0jhm0000gn/T/ipykernel_15696/4046326598.py:13: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


## run it

Runs the server on its own thread with its own event loop, so it doesn't fight
the notebook kernel's loop. Open http://127.0.0.1:8000

In [8]:
# uvicorn.Config/Server builds the server object but doesn't start it yet
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)

# a Jupyter kernel already has its own asyncio event loop running in the
# background, and uvicorn wants to run its own. instead of fighting over
# the same loop, we start a brand new thread and give the server a totally
# separate event loop to run in — daemon=True means it dies automatically
# when the notebook kernel shuts down
server_thread = threading.Thread(target=lambda: asyncio.run(server.serve()), daemon=True)
server_thread.start()

## evaluation

For each question in `eval/testset.json` we have a reference SQL query. We run
the agent, run the reference, and check the results match. Also reports how
many questions needed a retry, which shows the self-correction loop working.

In [9]:
import json

TESTSET_PATH = "eval/testset.json"


def result_matches(expected, got):
    # turns both lists of tuples into sets before comparing, so [(1,2),(3,4)]
    # matches [(3,4),(1,2)] too — row order from sql isn't guaranteed anyway
    if got is None:
        return False
    return set(expected) == set(got)


def run_eval():
    load_demo_data()   # eval questions are written against the demo dataset

    with open(TESTSET_PATH) as f:
        testset = json.load(f)   # list of {"question": ..., "sql": ...}

    conn = get_connection()

    correct = 0
    needed_retry = 0

    for item in testset:
        question = item["question"]
        # run the known-correct reference query ourselves, to compare against
        expected, _ = run_query(conn, item["sql"])

        # now let the agent answer the same question on its own
        result = ask(question)
        ok = result_matches(expected, result["rows"])

        if ok:
            correct += 1
        if result["attempts"] > 1:
            # more than 1 attempt means the first sql failed and the
            # retry loop kicked in and fixed it
            needed_retry += 1

        mark = "ok " if ok else "X  "
        print(f"{mark} (tries={result['attempts']}) {question}")

    conn.close()

    total = len(testset)
    print("\n--- results ---")
    print(f"accuracy: {correct}/{total} = {correct / total:.0%}")
    print(f"needed a retry: {needed_retry}/{total}")


# run_eval()  # uncomment to score the agent against the test set

INFO:     Started server process [15696]
